# Contents
The quantization tables are computed as in the previous cases, but I added 10 steps, so I get a total of 20.

In [ ]:
! pip install jpegio

In [ ]:
from PIL import Image
import numpy as np
import jpegio
import matplotlib.pyplot as plt
import os
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/tesi/" if 'google.colab' in str(get_ipython()) else "."
sample_jpeg_path = os.path.join(BASE_DIR, "experiments/tables/reference_image/000001.jpg")

In [ ]:
SAVE_PATH = os.path.join(BASE_DIR, "assets/increase_steps20_qt.pt")

## computing quantization tables

In [ ]:
qualities = [100, 45, 25, 15, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 1, 1, 1, 1, 1]

In [ ]:
current_img = Image.open(sample_jpeg_path)
quant_tables = {}

for i, q in enumerate(qualities):
    current_img.save(f'output_img_quality_{i}.jpg', quality=q)
    current_img = Image.open(f'output_img_quality_{i}.jpg').convert('RGB')
    jpeg_data = jpegio.read(f'output_img_quality_{i}.jpg')
    quant_tables[f'step{i}_luminance'] = np.array(jpeg_data.quant_tables[0])
    quant_tables[f'step{i}_chrominance'] = np.array(jpeg_data.quant_tables[1])

In [ ]:
TRANSITION_STEPS = 5
n_steps = len(qualities)

for idx in range(n_steps):
    for suffix in ['luminance', 'chrominance']:
        key = f'step{idx}_{suffix}'

        if 'quant_tables' in globals() and isinstance(quant_tables, dict) and key in quant_tables:
            value = torch.tensor(quant_tables[key], dtype=torch.float32).detach().clone()

            steps_from_end = n_steps - 1 - idx  # 0 = ultimo step, 1 = penultimo, ...

            if steps_from_end < TRANSITION_STEPS:
                # Apply the boosting logic for the last TRANSITION_STEPS
                # The boost factor increases as we get closer to the end.

                # Original boost: exponential (powers of 2)
                # boost = 2 ** (TRANSITION_STEPS - steps_from_end)

                # Option 1: Linear boost (lighter)
                # This creates a boost like: 1, 2, 3 for TRANSITION_STEPS = 3
                # boost = (TRANSITION_STEPS - steps_from_end) + 0.5

                # Option 2: Smaller base for exponential boost (e.g., base 1.5)
                # boost = 1.5 ** (TRANSITION_STEPS - steps_from_end)

                # Option 3: Additive boost (adds a constant or linearly increasing value)
                boost = 0.5 + (TRANSITION_STEPS - steps_from_end) * (TRANSITION_STEPS/10) # Example: adds 0.5, 1.0, 1.5

                value = value * boost

                # Assuming this specific modification is still desired for the boosted tables.
                if value.numel() > 0:
                    value[0, 0] = 1.0

            # Update the quantization table in the dictionary (now as a torch.Tensor)
            quant_tables[key] = value
        else:
            print(f"Key '{key}' not found in quant_tables or quant_tables is not defined/a dict.")

plotting images

In [ ]:
num_images = len(qualities)
plt.figure(figsize=(num_images * 4, 5))

for i in range(num_images):
    # saved_img = jpegio.read(f'output_img_quality_{qualities[i]}.jpg')
    saved_img = Image.open(f'output_img_quality_{i}.jpg').convert('RGB')
    plt.subplot(1, num_images, i + 1)
    plt.imshow(saved_img)
    plt.title(f'step {i}')
    plt.axis('off')

Output hidden; open in https://colab.research.google.com to view.

printing quantization tables

In [ ]:
for i in range(num_images):
    print(f"Luminance:\n{quant_tables[f'step{i}_luminance']}")
    print(f"Chrominance:\n{quant_tables[f'step{i}_chrominance']}")

Luminance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
Chrominance:
tensor([[1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])
Luminance:
tensor([[ 18.,  12.,  11.,  18.,  27.,  44.,  57.,  68.],
        [ 13.,  13.,  16.,  21.,  29.,  64.,  67.,  61.],
        [ 16.,  14.,  18.,  27.,  44.,  63.,  77.,  62.],
        [ 16.,  19.,  24.,  32.,  57.,  97.,  89.,  69.],
        [ 20.,  24.,  41.,  62.,  75., 121., 114.,  85.],
 

# saving quantization tables

### pt

In [ ]:
import torch

torch.save(quant_tables, SAVE_PATH)
print(f"Quantization tables saved to: {SAVE_PATH}")

Quantization tables saved to: /content/drive/MyDrive/Colab Notebooks/tesi/assets/increase_steps20_qt.pt


In [ ]:
quant_tables = torch.load(SAVE_PATH)

# Accesso diretto, già tensori
lum = quant_tables['step10_luminance']#.to('cuda')
chrom = quant_tables['step10_chrominance']#.to('cuda')

In [ ]:
print(f"Luminance:\n{lum}")
print(f"Chrominance:\n{chrom}")